# Chest X-ray14: Comparacion DenseNet121 vs ResNet vs ViT-B/16

Clasificacion multi-etiqueta (14 hallazgos radiologicos) sobre NIH ChestX-ray14. Ver `PLAN.md` para el contexto completo del proyecto (por que estas decisiones, cronograma, criterios de evaluacion).

**Este notebook se escribio en una maquina sin GPU y sin el dataset.** Fue disenado para correr de principio a fin en la maquina donde si esta `archive.zip` (dataset de Kaggle, `nih-chest-xrays/data`) y hay GPU disponible.

**Requisito:** coloca `archive.zip` en la raiz de este proyecto (junto a este notebook) antes de correr.

## 1. Setup

In [ ]:
import os
import json
import time
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.utils.data as data
from torchvision import models, transforms
from sklearn.metrics import f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PROJECT_ROOT = Path(".").resolve()
ARCHIVE_PATH = PROJECT_ROOT / "archive.zip"
IMAGES_DIR = PROJECT_ROOT / "images_subset"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

assert ARCHIVE_PATH.exists(), (
    f"No se encontro {ARCHIVE_PATH}. Coloca archive.zip (dataset de Kaggle nih-chest-xrays/data) "
    "en la raiz del proyecto antes de continuar."
)

# ---- Configuracion del experimento (ajustable) ----
N_SAMPLES = 12000          # tamano del subconjunto a extraer del dataset completo (112k imagenes)
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # el resto (0.15) es test

DISEASE_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(DISEASE_LABELS)
print(f"{NUM_CLASSES} clases:", DISEASE_LABELS)


## 2. Datos

`archive.zip` pesa ~42GB (112k imagenes) — no lo extraemos completo. En su lugar:
1. Leemos el CSV de etiquetas directamente de dentro del zip (sin extraer nada).
2. Muestreamos `N_SAMPLES` imagenes al azar (semilla fija).
3. Extraemos **solo esas imagenes** a `images_subset/`.

El codigo busca el CSV y las imagenes por nombre de archivo dentro del zip, sin asumir una ruta interna exacta (la estructura oficial del dataset de Kaggle tiene subcarpetas `images_001/images/` ... `images_012/images/`, pero esto lo hace robusto si la estructura real difiere un poco).

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    names = zf.namelist()
    print(f"Total de entradas en el zip: {len(names)}")

    # Encontrar el CSV de etiquetas (normalmente 'Data_Entry_2017.csv')
    csv_candidates = [n for n in names if n.lower().endswith(".csv") and "data_entry" in n.lower()]
    if not csv_candidates:
        csv_candidates = [n for n in names if n.lower().endswith(".csv")]
    assert csv_candidates, "No se encontro ningun CSV de etiquetas dentro de archive.zip. Revisa el contenido del zip manualmente."
    labels_csv_name = csv_candidates[0]
    print("CSV de etiquetas encontrado:", labels_csv_name)

    with zf.open(labels_csv_name) as f:
        df_labels = pd.read_csv(f)

print(df_labels.shape)
df_labels.head()


In [ ]:
# Columna con el nombre de archivo de imagen (varia ligeramente segun la version del CSV)
image_col_candidates = [c for c in df_labels.columns if "image" in c.lower() and "index" in c.lower()]
IMAGE_COL = image_col_candidates[0] if image_col_candidates else df_labels.columns[0]
LABEL_COL = [c for c in df_labels.columns if "finding" in c.lower() and "label" in c.lower()][0]
print("Columna de imagen:", IMAGE_COL, "| Columna de etiquetas:", LABEL_COL)

# Muestreo aleatorio de N_SAMPLES filas (con semilla fija para reproducibilidad)
df_sample = df_labels.sample(n=min(N_SAMPLES, len(df_labels)), random_state=SEED).reset_index(drop=True)
print(f"Subconjunto muestreado: {len(df_sample)} imagenes")

# Vector binario de 14 clases a partir de 'Finding Labels' (pipe-separated; 'No Finding' = todo ceros)
def labels_to_vector(finding_labels_str):
    present = set(finding_labels_str.split("|"))
    return np.array([1.0 if label in present else 0.0 for label in DISEASE_LABELS], dtype=np.float32)

df_sample["label_vector"] = df_sample[LABEL_COL].apply(labels_to_vector)
print("Ejemplo:", df_sample[IMAGE_COL].iloc[0], "->", df_sample[LABEL_COL].iloc[0], "->", df_sample["label_vector"].iloc[0])


In [ ]:
# Extraer solo las N_SAMPLES imagenes necesarias del zip (busca por nombre de archivo, sin asumir subcarpeta exacta)
wanted_filenames = set(df_sample[IMAGE_COL])
extracted, missing = 0, []

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    name_by_basename = {Path(n).name: n for n in zf.namelist() if Path(n).name in wanted_filenames}
    for fname in wanted_filenames:
        dest = IMAGES_DIR / fname
        if dest.exists():
            extracted += 1
            continue
        internal_name = name_by_basename.get(fname)
        if internal_name is None:
            missing.append(fname)
            continue
        with zf.open(internal_name) as src, open(dest, "wb") as out:
            out.write(src.read())
        extracted += 1

print(f"Extraidas: {extracted} | Faltantes: {len(missing)}")
if missing:
    print("Ejemplos de imagenes no encontradas en el zip (revisar nombres):", missing[:5])
    df_sample = df_sample[~df_sample[IMAGE_COL].isin(missing)].reset_index(drop=True)
    print(f"Continuando con {len(df_sample)} imagenes validas.")


In [ ]:
class ChestXray14Dataset(data.Dataset):
    def __init__(self, df, images_dir, image_col, transform):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.image_col = image_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.images_dir / row[self.image_col]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        label = torch.from_numpy(row["label_vector"])
        return image, label


In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

n = len(df_sample)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)
df_shuffled = df_sample.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df_train = df_shuffled.iloc[:n_train]
df_val = df_shuffled.iloc[n_train:n_train + n_val]
df_test = df_shuffled.iloc[n_train + n_val:]
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

train_dataset = ChestXray14Dataset(df_train, IMAGES_DIR, IMAGE_COL, train_transform)
val_dataset = ChestXray14Dataset(df_val, IMAGES_DIR, IMAGE_COL, eval_transform)
test_dataset = ChestXray14Dataset(df_test, IMAGES_DIR, IMAGE_COL, eval_transform)

train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
# Vistazo rapido a un batch, para confirmar que el pipeline de datos esta correcto antes de entrenar nada
imgs, labels = next(iter(train_loader))
print("Batch de imagenes:", imgs.shape, "| Batch de etiquetas:", labels.shape)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    present = [DISEASE_LABELS[j] for j, v in enumerate(labels[i]) if v == 1]
    ax.imshow(img)
    ax.set_title(", ".join(present) if present else "No Finding", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_batch.png", dpi=120)
plt.show()


## 3. Modelos

DenseNet121, ResNet18 y ViT-B/16, todos preentrenados en ImageNet (`torchvision.models`). El backbone queda congelado salvo el ultimo bloque, y se reemplaza la capa final por una de 14 salidas (clasificacion multi-etiqueta, sin softmax — se usa `BCEWithLogitsLoss` mas adelante).

In [ ]:
def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def build_model(name):
    """Regresa un modelo preentrenado con head de NUM_CLASES salidas y backbone mayormente congelado."""
    if name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, NUM_CLASSES)
        for p in model.features.denseblock4.parameters():
            p.requires_grad = True
        for p in model.features.norm5.parameters():
            p.requires_grad = True

    elif name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        for p in model.layer4.parameters():
            p.requires_grad = True

    elif name == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, NUM_CLASSES)
        for p in model.encoder.layers[-1].parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Modelo desconocido: {name}")

    return model.to(device)


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


MODEL_NAMES = ["densenet121", "resnet18", "vit_b_16"]
for name in MODEL_NAMES:
    m = build_model(name)
    total, trainable = count_trainable_params(m)
    print(f"{name}: {total:,} parametros totales, {trainable:,} entrenables")
    del m


## 4. Entrenamiento

Mismo protocolo experimental para los 3 modelos: mismo split, misma resolucion, mismo numero de epocas, mismo optimizador. El loop esta adaptado del patron `run_epoch`/`fit` visto en `CNNs_Fundamentals_PyTorch.ipynb`, cambiando a `BCEWithLogitsLoss` para clasificacion multi-etiqueta.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_batches = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = criterion(preds, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    history = {"train_loss": [], "val_loss": [], "epoch_time_sec": []}
    for epoch in range(epochs):
        t0 = time.time()
        train_loss = run_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_epoch(model, val_loader, criterion, optimizer=None)
        epoch_time = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["epoch_time_sec"].append(epoch_time)
        print(f"  epoch {epoch+1}/{epochs} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - {epoch_time:.1f}s")

    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else 0.0
    )
    return history, peak_memory_mb


In [ ]:
trained_models = {}
training_results = {}

for name in MODEL_NAMES:
    print(f"\n=== Entrenando {name} ===")
    model = build_model(name)
    total_params, trainable_params = count_trainable_params(model)
    history, peak_memory_mb = fit(model, train_loader, val_loader)

    trained_models[name] = model
    training_results[name] = {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "history": history,
        "peak_memory_mb": peak_memory_mb,
        "avg_epoch_time_sec": float(np.mean(history["epoch_time_sec"])),
    }


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name in MODEL_NAMES:
    ax.plot(training_results[name]["history"]["val_loss"], marker="o", label=name)
ax.set_xlabel("Epoca")
ax.set_ylabel("Val loss (BCE)")
ax.set_title("Curva de convergencia por modelo")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "val_loss_curves.png", dpi=120)
plt.show()


## 5. Evaluacion

Accuracy promedio por etiqueta y F1 (macro/micro) sobre el test set, para cada modelo.

In [ ]:
def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.numpy())
    y_prob = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    y_pred = (y_prob >= threshold).astype(np.float32)

    per_label_accuracy = (y_pred == y_true).mean()
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    return {"accuracy": float(per_label_accuracy), "f1_macro": float(f1_macro), "f1_micro": float(f1_micro)}


eval_results = {}
for name in MODEL_NAMES:
    metrics = evaluate_model(trained_models[name], test_loader)
    eval_results[name] = metrics
    print(f"{name}: accuracy={metrics['accuracy']:.4f}  F1_macro={metrics['f1_macro']:.4f}  F1_micro={metrics['f1_micro']:.4f}")

comparison_df = pd.DataFrame({
    "Modelo": MODEL_NAMES,
    "Accuracy": [eval_results[n]["accuracy"] for n in MODEL_NAMES],
    "F1 macro": [eval_results[n]["f1_macro"] for n in MODEL_NAMES],
    "F1 micro": [eval_results[n]["f1_micro"] for n in MODEL_NAMES],
    "Parametros totales": [training_results[n]["total_params"] for n in MODEL_NAMES],
    "Parametros entrenables": [training_results[n]["trainable_params"] for n in MODEL_NAMES],
    "Tiempo/epoca (s)": [training_results[n]["avg_epoch_time_sec"] for n in MODEL_NAMES],
    "Memoria pico (MB)": [training_results[n]["peak_memory_mb"] for n in MODEL_NAMES],
})
comparison_df


## 6. Interpretabilidad

Grad-CAM para los modelos convolucionales (DenseNet121, ResNet18) y mapa de atencion del token `[CLS]` hacia los patches para ViT-B/16.

In [ ]:
# Grad-CAM para DenseNet121 y ResNet18. Si 'grad-cam' no esta instalado: pip install grad-cam
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    GRADCAM_AVAILABLE = True
except ImportError:
    print("pytorch-grad-cam no instalado -- corre 'pip install grad-cam' para habilitar esta celda.")
    GRADCAM_AVAILABLE = False

example_img, example_label = test_dataset[0]
input_tensor = example_img.unsqueeze(0).to(device)

unnorm_img = example_img.permute(1, 2, 0).numpy()
unnorm_img = unnorm_img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
unnorm_img = np.clip(unnorm_img, 0, 1)

gradcam_target_layers = {
    "densenet121": lambda m: [m.features.norm5],
    "resnet18": lambda m: [m.layer4[-1]],
}

if GRADCAM_AVAILABLE:
    fig, axes = plt.subplots(1, len(gradcam_target_layers), figsize=(5 * len(gradcam_target_layers), 5))
    for ax, (name, layer_fn) in zip(np.atleast_1d(axes), gradcam_target_layers.items()):
        model = trained_models[name]
        cam = GradCAM(model=model, target_layers=layer_fn(model))
        grayscale_cam = cam(input_tensor=input_tensor)[0]
        overlay = show_cam_on_image(unnorm_img, grayscale_cam, use_rgb=True)
        ax.imshow(overlay)
        ax.set_title(f"Grad-CAM: {name}")
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "gradcam_examples.png", dpi=120)
    plt.show()


In [ ]:
# Mapa de atencion para ViT-B/16: atencion del token [CLS] hacia los patches en el ultimo bloque del encoder.
# torchvision no expone los pesos de atencion directamente en el forward normal, asi que reproducimos
# manualmente los primeros pasos del forward (patch embedding + pos encoding + encoder) hasta el ultimo bloque.

vit_model = trained_models["vit_b_16"]
vit_model.eval()

with torch.no_grad():
    x = vit_model._process_input(input_tensor)
    n = x.shape[0]
    batch_class_token = vit_model.class_token.expand(n, -1, -1)
    x = torch.cat([batch_class_token, x], dim=1)
    x = x + vit_model.encoder.pos_embedding
    x = vit_model.encoder.dropout(x)

    # Pasar por todos los bloques salvo el ultimo normalmente
    for layer in vit_model.encoder.layers[:-1]:
        x = layer(x)

    # En el ultimo bloque, llamar la self-attention directamente para obtener los pesos
    last_block = vit_model.encoder.layers[-1]
    x_norm = last_block.ln_1(x)
    _, attn_weights = last_block.self_attention(
        x_norm, x_norm, x_norm, need_weights=True, average_attn_weights=True
    )
    # attn_weights: [batch, seq_len, seq_len]; fila 0 = atencion del CLS token hacia todos los tokens (incluido el mismo)
    cls_attention = attn_weights[0, 0, 1:].cpu().numpy()  # excluye el propio CLS

    grid_size = int(np.sqrt(cls_attention.shape[0]))
    attention_map = cls_attention.reshape(grid_size, grid_size)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(unnorm_img)
axes[0].set_title("Imagen original")
axes[0].axis("off")
axes[1].imshow(unnorm_img)
axes[1].imshow(attention_map, cmap="jet", alpha=0.5, extent=(0, 1, 1, 0))
axes[1].set_title("Atencion del token [CLS] (ViT-B/16, ultimo bloque)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "attention_example.png", dpi=120)
plt.show()


## 7. Exportar resultados

`results/metrics.json` queda como la fuente unica de verdad para el paper y las slides: no hace falta transcribir numeros a mano ni volver a entrenar para armar tablas.

In [ ]:
final_results = {
    "config": {
        "n_samples": N_SAMPLES,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "train_frac": TRAIN_FRAC,
        "val_frac": VAL_FRAC,
        "seed": SEED,
        "device": str(device),
    },
    "models": {},
}

for name in MODEL_NAMES:
    final_results["models"][name] = {
        **eval_results[name],
        "total_params": training_results[name]["total_params"],
        "trainable_params": training_results[name]["trainable_params"],
        "avg_epoch_time_sec": training_results[name]["avg_epoch_time_sec"],
        "peak_memory_mb": training_results[name]["peak_memory_mb"],
        "train_loss_history": training_results[name]["history"]["train_loss"],
        "val_loss_history": training_results[name]["history"]["val_loss"],
    }

with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2)

print("Resultados guardados en", RESULTS_DIR / "metrics.json")
comparison_df
